# Multi-Object Detection & Persistent ID Tracking
## Full Demo Notebook — All Optional Enhancements

**This notebook covers:**
1. Run the tracking pipeline
2. Trajectory visualisation
3. Movement heatmap
4. Person count over time
5. Speed estimation
6. Team / role clustering
7. Bird's-eye projection
8. Evaluation metrics
9. Model comparison (YOLOv8n vs YOLOv8s)


In [ ]:
import sys, os
sys.path.insert(0, 'src')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.grid']  = True
plt.rcParams['grid.alpha'] = 0.3

TRACK_CSV = 'data/output/track_data.csv'
COUNT_CSV = 'data/output/count_over_time.csv'
VIDEO     = 'data/input/video.mp4'
OUT_DIR   = 'data/output'

print('✅ Setup complete')

## Step 1 — Run Tracking Pipeline

In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'src/main.py', '--input', VIDEO,
     '--output', 'data/output/output.mp4',
     '--no-display', '--frame-skip', '2'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 2 — Load Track Data

In [ ]:
df_tracks = pd.read_csv(TRACK_CSV)
df_counts = pd.read_csv(COUNT_CSV)

print(f'Tracks shape : {df_tracks.shape}')
print(f'Unique IDs   : {df_tracks.track_id.nunique()}')
print(f'Frame range  : {df_tracks.frame.min()} – {df_tracks.frame.max()}')
df_tracks.head()

## Step 3 — Trajectory Visualisation

In [ ]:
from visualize import get_color

# Load last frame as background
cap = cv2.VideoCapture(VIDEO)
cap.set(cv2.CAP_PROP_POS_AVI_RATIO, 1)
ret, bg = cap.read()
cap.release()
if not ret:
    bg = np.zeros((540, 960, 3), np.uint8)

traj_img = (bg * 0.3).astype(np.uint8)
for tid, grp in df_tracks.groupby('track_id'):
    pts = grp[['cx','cy']].values
    color_bgr = get_color(tid)
    color_rgb = (color_bgr[2]/255, color_bgr[1]/255, color_bgr[0]/255)
    for i in range(1, len(pts)):
        cv2.line(traj_img, tuple(pts[i-1]), tuple(pts[i]), color_bgr, 2)

plt.figure(figsize=(14, 7))
plt.imshow(cv2.cvtColor(traj_img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Full Trajectory Map — All Tracked Players', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/trajectory_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: trajectory_map.png')

## Step 4 — Movement Heatmap

In [ ]:
heatmap_img = cv2.imread(f'{OUT_DIR}/heatmap.png')
if heatmap_img is not None:
    plt.figure(figsize=(14, 7))
    plt.imshow(cv2.cvtColor(heatmap_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Movement Heatmap (foot positions across all frames)', fontsize=14)
    plt.tight_layout()
    plt.show()

# Also make a pure heatmap without background
h = int(bg.shape[0])
w = int(bg.shape[1])
canvas = np.zeros((h, w), np.float32)
for _, row in df_tracks.iterrows():
    fx, fy = int(row.cx), int(row.y2)
    if 0 <= fy < h and 0 <= fx < w:
        cv2.circle(canvas, (fx, fy), 20, 1.0, -1)

norm = cv2.normalize(canvas, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
plt.figure(figsize=(14, 6))
plt.imshow(norm, cmap='hot', interpolation='bilinear')
plt.colorbar(label='Presence density')
plt.title('Pure Density Heatmap', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

## Step 5 — Person Count Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Raw count
axes[0].plot(df_counts.frame, df_counts.person_count,
             lw=1, color='#2196F3', alpha=0.7, label='Raw count')
roll = df_counts.person_count.rolling(30, center=True).mean()
axes[0].plot(df_counts.frame, roll,
             lw=2.5, color='#F44336', label='30-frame rolling avg')
axes[0].fill_between(df_counts.frame, df_counts.person_count, alpha=0.1, color='#2196F3')
axes[0].set_ylabel('People detected')
axes[0].set_title('Person Count Over Time')
axes[0].legend()

# Count distribution
axes[1].hist(df_counts.person_count, bins=range(0, df_counts.person_count.max()+2),
             color='#4CAF50', edgecolor='white', rwidth=0.8)
axes[1].set_xlabel('Person count')
axes[1].set_ylabel('Frequency (frames)')
axes[1].set_title('Distribution of Detection Counts')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/count_plot.png', dpi=150)
plt.show()
print(f"Stats: max={df_counts.person_count.max()}, "
      f"avg={df_counts.person_count.mean():.1f}, "
      f"median={df_counts.person_count.median():.0f}")

## Step 6 — Speed Estimation

In [ ]:
from speed_estimation import compute_speeds

cap = cv2.VideoCapture(VIDEO)
fps = max(1, int(cap.get(cv2.CAP_PROP_FPS)))
cap.release()

track_data = df_tracks.to_dict('records')
# pixels_per_metre=None → results in px/s
# For real calibration set e.g. pixels_per_metre=8.5
stats = compute_speeds(track_data, fps, pixels_per_metre=None)

df_speed = pd.DataFrame([
    {'track_id': tid, **s} for tid, s in stats.items()
]).sort_values('avg_speed', ascending=False)

print(df_speed.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df_speed.head(10).plot.barh('track_id', 'avg_speed', ax=axes[0],
                             color='#FF5722', legend=False)
axes[0].set_xlabel('Avg speed (px/s)')
axes[0].set_title('Top 10 Players by Average Speed')
axes[0].invert_yaxis()

axes[1].scatter(df_speed.avg_speed, df_speed.max_speed,
                c='#9C27B0', alpha=0.7, s=80)
axes[1].set_xlabel('Average speed (px/s)')
axes[1].set_ylabel('Peak speed (px/s)')
axes[1].set_title('Avg vs Peak Speed per Track')
for _, row in df_speed.iterrows():
    axes[1].annotate(str(int(row.track_id)),
                     (row.avg_speed, row.max_speed), fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/speed_chart.png', dpi=150)
plt.show()

## Step 7 — Team Clustering

In [ ]:
from team_clustering import TeamClusterer, TEAM_LABELS, TEAM_COLORS

cap = cv2.VideoCapture(VIDEO)
clusterer = TeamClusterer(min_samples_to_cluster=5)

by_frame = df_tracks.groupby('frame')
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_count += 1
    if frame_count not in by_frame.groups: continue
    for _, row in by_frame.get_group(frame_count).iterrows():
        clusterer.update(int(row.track_id), frame,
                         int(row.x1), int(row.y1), int(row.x2), int(row.y2))
    if frame_count % 60 == 0:
        clusterer.assign_teams()

cap.release()
clusterer.assign_teams()

labels = clusterer._labels
df_tracks['team'] = df_tracks.track_id.map(lambda tid: TEAM_LABELS.get(labels.get(tid, -1), '?'))

print('Team assignments:')
print(df_tracks.groupby(['track_id','team']).size().reset_index(name='frames').to_string(index=False))

# Visualise team split
team_counts = df_tracks.groupby('team').track_id.nunique()
colors = ['#2196F3', '#FF5722', '#9E9E9E']
plt.figure(figsize=(6, 4))
team_counts.plot.bar(color=colors[:len(team_counts)], edgecolor='white')
plt.ylabel('Unique player IDs')
plt.title('Players per Team (KMeans jersey clustering)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/team_split.png', dpi=150)
plt.show()

## Step 8 — Evaluation Metrics

In [ ]:
from evaluation import compute_tracklet_metrics, print_metrics, plot_lifetime_histogram

data    = df_tracks.to_dict('records')
metrics = compute_tracklet_metrics(data)
print_metrics(metrics)

lifetimes = metrics['track_lifetimes']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(lifetimes, bins=20, color='#4CAF50', edgecolor='white')
axes[0].set_xlabel('Track lifetime (frames)')
axes[0].set_ylabel('Number of tracks')
axes[0].set_title('Track Lifetime Distribution')

axes[1].bar(range(len(lifetimes)), sorted(lifetimes, reverse=True),
            color='#2196F3', width=1.0)
axes[1].set_xlabel('Track rank (longest first)')
axes[1].set_ylabel('Duration (frames)')
axes[1].set_title('Track Duration by Rank')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/track_lifetime_plot.png', dpi=150)
plt.show()

## Step 9 — Model Comparison: YOLOv8n vs YOLOv8s

In [ ]:
# ⚠️ This cell downloads yolov8s.pt (~22 MB) on first run
from evaluation import compare_models
compare_models(
    VIDEO,
    n_frames=80,
    out_path=f'{OUT_DIR}/model_comparison.png'
)
img = cv2.imread(f'{OUT_DIR}/model_comparison.png')
if img is not None:
    plt.figure(figsize=(15, 4))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

## Summary of All Outputs

In [ ]:
print('📁 Files in data/output/')
print('-' * 50)
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    icon = '🎬' if f.endswith('.mp4') else '🖼️' if f.endswith('.png') else '📊'
    print(f'  {icon}  {f:<40} {size/1024:>7.0f} KB')